# NeuroTrain Lab — Notebook 2: Loss and Backpropagation

**Topic:** what a loss function is, how MSE (regression) and Cross-Entropy
(classification) measure error, and how the chain rule lets us calculate
exactly how much each weight should change to reduce that error — what
libraries call *backpropagation*.

> Second of 4 notebooks. In Notebook 1 you learned to **predict** (forward
> propagation). Here you learn to **measure the error** and compute the
> direction each weight should move to reduce it. Actually moving them —
> training — is Notebook 3.

## 🎯 What you'll learn in this notebook

By the end you should be able to explain, without memorized formulas:

1. What a loss function is and why it isn't the same thing as accuracy.
2. How MSE (Mean Squared Error) works for regression problems.
3. How Binary Cross-Entropy works for classification problems, and why "being
   confidently wrong" is penalized far more than "being unsure."
4. What the chain rule is and how it's applied, step by step, over a small
   computational graph to get a gradient.
5. That `loss.backward()` in PyTorch isn't magic — it's exactly the same
   arithmetic you just did by hand.

**Mental map:** `prediction → loss → chain rule → gradient → (Notebook 3: update weights)`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| PyTorch:", torch.__version__, "| TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## 1. What a loss function is

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — The 'hot / cold' game</b><br><br>
As kids we'd hide an object and guide someone with "cold" (far) or "hot"
(close). A loss function does exactly that: it turns "how wrong is this
prediction" into **a single number**. The higher it is, the "colder" the model
is; the lower (ideally 0), the "hotter."

It doesn't tell us **which direction** to move — that's the chain rule, in
Section 4 — but it does tell us whether we're improving.
</div>

### Loss vs. accuracy

These are not the same thing, and confusing them is a typical mistake:

- The **loss** is what the optimizer **directly minimizes**. It's continuous
  and sensitive: it distinguishes "I got it right with 0.51 probability" from
  "I got it right with 0.99 probability," even though both count as correct.
- **Accuracy** is what's easy for humans to interpret ("it got 90% right"), but
  it's a cruder count — it only looks at whether you crossed the 0.5 threshold,
  not by how much.

That's why they can **diverge short-term**: loss can go down (the model is
getting more confident) without accuracy changing yet, or vice versa.

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — So why not just train directly to maximize accuracy?</b><br><br>
Because accuracy isn't **differentiable**: it's a step function (right/wrong),
not a smooth curve. The chain rule (Section 4) needs a smooth function to
calculate which direction to move each weight. That's why we train by
minimizing a continuous loss (MSE, Cross-Entropy...) and use accuracy only to
**interpret** the result, not to optimize it.
</div>

## 2. MSE (Mean Squared Error): the loss for regression

For a **regression** problem (predicting a continuous number, not a class), the
most common loss is the **Mean Squared Error**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Squaring does two things: (1) negative and positive errors don't cancel each
other out, and (2) it penalizes large errors far more than small ones.

In [ ]:
# Toy example: predicting the real temperature (°C) from a cheap thermometer
y_true = np.array([20.0, 25.0, 30.0, 22.0])
y_pred = np.array([18.0, 27.0, 29.0, 25.0])

squared_errors = (y_true - y_pred) ** 2
mse = squared_errors.mean()

print("Squared errors:", squared_errors)
print("MSE:", mse)

plt.figure(figsize=(5.5, 4))
x_pos = np.arange(len(y_true))
plt.scatter(x_pos, y_true, color="#2563EB", label="Real", zorder=3, s=60)
plt.scatter(x_pos, y_pred, color="#F97316", label="Prediction", zorder=3, s=60)
for xi, real, pred in zip(x_pos, y_true, y_pred):
    plt.plot([xi, xi], [real, pred], color="#94A3B8", linestyle="--", zorder=1)
plt.title(f"Squared gap between prediction and reality (MSE = {mse:.2f})")
plt.xticks(x_pos, [f"reading {i+1}" for i in x_pos])
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### ✏️ Exercise

Implement `mse(y_true, y_pred)` manually (without using `tf.keras.losses`) and
check it matches `tf.keras.losses.MeanSquaredError()` on the same
`y_true`/`y_pred` above (should give `4.5`).

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ✏️✏️✏️)


my_mse = mse(y_true, y_pred)
mse_keras = tf.keras.losses.MeanSquaredError()(y_true, y_pred).numpy()

print("Manual:", my_mse, "| Keras:", mse_keras)
assert np.isclose(my_mse, mse_keras)
print("They match!")

<details>
<summary><b>Show solution</b></summary>

```python
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


my_mse = mse(y_true, y_pred)
mse_keras = tf.keras.losses.MeanSquaredError()(y_true, y_pred).numpy()

print("Manual:", my_mse, "| Keras:", mse_keras)
assert np.isclose(my_mse, mse_keras)
print("They match!")
```

</details>

## 3. Cross-Entropy: the loss for classification

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — MSE isn't the natural fit for our problem</b><br><br>
Our running problem — benign or malignant tumor? — is **binary
classification**: the output is a probability between 0 and 1, not an
unbounded continuous number. MSE would treat "I predicted 0.5 when it was 1"
and "I predicted 0.99 when it was 0" too similarly in relative terms — it
doesn't capture that **being confident and wrong is far worse than being
unsure**. That's what **Binary Cross-Entropy (BCE)** is for.
</div>

For a true label $y \in \{0, 1\}$ and a predicted probability $p$:

$$\text{BCE} = -\big[y \log(p) + (1-y)\log(1-p)\big]$$

If the true label is $y=1$, this reduces to $-\log(p)$: the further $p$ is from
1, the larger (and faster-growing) the loss.

In [ ]:
# Numeric table: y_true = 1 (malignant), three confidence levels
bce_keras = tf.keras.losses.BinaryCrossentropy()
y_true_bce = tf.constant([[1.0]])

probabilities = [0.9, 0.5, 0.1]
losses = [bce_keras(y_true_bce, tf.constant([[p]])).numpy() for p in probabilities]

for p, l in zip(probabilities, losses):
    print(f"p={p:>3} ({'correct' if p > 0.5 else 'incorrect' if p < 0.5 else 'no'} confidence)"
          f"  ->  BCE = {l:.4f}")

plt.figure(figsize=(5, 3.8))
plt.bar([str(p) for p in probabilities], losses, color=["#22C55E", "#F97316", "#F43F5E"])
plt.title("BCE for y=1 as a function of the predicted probability")
plt.xlabel("predicted p")
plt.ylabel("Binary Cross-Entropy")
plt.grid(alpha=0.2, axis="y")
plt.show()

With `y=1`: `p=0.9` (correct and confident) gives BCE ≈ **0.105**; `p=0.5`
(unsure) gives BCE ≈ **0.693**; `p=0.1` (confident and **wrong**) gives BCE ≈
**2.303** — more than 20 times the loss at `p=0.9`. That asymmetry is
intentional: the model learns **not to be confidently wrong**.

### ✏️ Exercise

Implement `binary_cross_entropy(y_true, y_pred)` manually using
`-[y·log(p) + (1-y)·log(1-p)]`, and check it matches
`tf.keras.losses.BinaryCrossentropy()` for `y=1, p=0.9`.

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(✏️✏️✏️))


my_bce = binary_cross_entropy(np.array([1.0]), np.array([0.9]))
bce_keras_val = bce_keras(tf.constant([[1.0]]), tf.constant([[0.9]])).numpy()

print("Manual:", my_bce, "| Keras:", bce_keras_val)
assert np.isclose(my_bce, bce_keras_val, atol=1e-5)
print("They match!")

<details>
<summary><b>Show solution</b></summary>

```python
def binary_cross_entropy(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


my_bce = binary_cross_entropy(np.array([1.0]), np.array([0.9]))
bce_keras_val = bce_keras(tf.constant([[1.0]]), tf.constant([[0.9]])).numpy()

print("Manual:", my_bce, "| Keras:", bce_keras_val)
assert np.isclose(my_bce, bce_keras_val, atol=1e-5)
print("They match!")
```

</details>

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ TYPICAL MISTAKE — np.log(0) breaks everything</b><br><br>
If `p` ever hits exactly 0 or 1, `np.log(0)` gives `-inf` and the loss explodes.
Keras avoids this internally by clipping `p` to a safe range (e.g.
`[1e-7, 1-1e-7]`). We don't need it in these examples because we chose `p`
strictly between 0 and 1, but it's why you should never use Sigmoid + manual
BCE without that clipping in production code.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🔥 Halfway there: from here it's a short hop to unravelling the mysteries of the artificial mind.</div>

## 4. The chain rule: how a gradient is computed

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — The heart of this notebook</b><br><br>
We already know how to **measure** error (Sections 2-3). Now we need to know
**which direction and by how much** to move each weight to reduce it. That's a
**gradient**: the derivative of the loss with respect to a weight, `dL/dw`. The
chain rule lets us compute it step by step, multiplying local derivatives along
the path from the loss back to the weight — that's *backpropagation*.
</div>

### Minimal example: one input, one weight

$$x = 2.0 \qquad w = 0.5 \qquad \text{pred} = x \cdot w \qquad \text{target} = 2.0
\qquad L = (\text{pred} - \text{target})^2$$

Computational graph (forward):

```
w --(× x)--> pred --(vs target)--> L
```

And backward (what we want: `dL/dw`):

```
L --> pred --> w
```

Step by step, with the chain rule `dL/dw = (dL/dpred) · (dpred/dw)`:

1. `pred = x · w = 2.0 · 0.5 = 1.0`
2. `L = (pred - target)² = (1.0 - 2.0)² = 1.0`
3. `dL/dpred = 2·(pred - target) = 2·(-1.0) = -2.0`
4. `dpred/dw = x = 2.0`
5. `dL/dw = dL/dpred · dpred/dw = -2.0 · 2.0 = -4.0`

`dL/dw = -4.0` means: if we increase `w` a little, the loss **decreases**
(negative gradient) — so the optimizer (Notebook 3) will move `w` in the
**opposite** direction of the gradient to reduce `L`.

In [ ]:
# Verify the example above in code, no autograd yet — pure manual computation
x, w, target = 2.0, 0.5, 2.0

pred = x * w
loss = (pred - target) ** 2

dL_dpred = 2 * (pred - target)
dpred_dw = x
dL_dw = dL_dpred * dpred_dw

print(f"pred={pred}  loss={loss}")
print(f"dL/dpred={dL_dpred}  dpred/dw={dpred_dw}  dL/dw={dL_dw}")

### A slightly bigger graph: input → linear → activation → loss

We add a bias and a ReLU activation before computing the loss — exactly the
structure of a real neuron followed by its loss:

$$x=3.0 \quad w=0.4 \quad b=-0.5 \quad z = x\cdot w + b \quad \text{pred} = \text{ReLU}(z)
\quad \text{target}=1.5 \quad L=(\text{pred}-\text{target})^2$$

Forward, step by step:

1. `z = x·w + b = 3.0·0.4 + (-0.5) = 0.7`
2. `pred = ReLU(z) = ReLU(0.7) = 0.7` (z is positive, ReLU leaves it unchanged)
3. `L = (pred - target)² = (0.7 - 1.5)² = 0.64`

Backward, step by step (chain rule at each node, back to front):

4. `dL/dpred = 2·(pred - target) = 2·(-0.8) = -1.6`
5. `dpred/dz = 1` if `z > 0`, `0` if `z < 0` (ReLU's derivative) → here `z=0.7>0`, so `dpred/dz = 1`
6. `dL/dz = dL/dpred · dpred/dz = -1.6 · 1 = -1.6`
7. `dz/dw = x = 3.0` → `dL/dw = dL/dz · dz/dw = -1.6 · 3.0 = -4.8`
8. `dz/db = 1` → `dL/db = dL/dz · dz/db = -1.6`

In [ ]:
# Same graph (linear + ReLU + loss), verified in code
def relu(z):
    return max(0.0, z)


x2, w2, b2, target2 = 3.0, 0.4, -0.5, 1.5

z2 = x2 * w2 + b2
pred2 = relu(z2)
loss2 = (pred2 - target2) ** 2

dL_dpred2 = 2 * (pred2 - target2)
dpred_dz2 = 1.0 if z2 > 0 else 0.0
dL_dz2 = dL_dpred2 * dpred_dz2
dL_dw2 = dL_dz2 * x2
dL_db2 = dL_dz2 * 1.0

print(f"z={z2}  pred={pred2}  loss={loss2}")
print(f"dL/dpred={dL_dpred2}  dpred/dz={dpred_dz2}  dL/dz={dL_dz2}")
print(f"dL/dw={dL_dw2}  dL/db={dL_db2}")

### ✏️ Exercise

Repeat the same graph (`z = x·w + b`, `pred = ReLU(z)`, `L = (pred - target)²`)
with new numbers: `x=1.5, w=0.3, b=0.2, target=0.5`. Fill in the forward pass
(`z3` and `pred3`) — the rest (backward) is already written so you can check
your result.

In [ ]:
x3, w3, b3, target3 = 1.5, 0.3, 0.2, 0.5

z3 = ✏️✏️✏️
pred3 = relu(z3)
loss3 = (pred3 - target3) ** 2

dL_dpred3 = 2 * (pred3 - target3)
dpred_dz3 = 1.0 if z3 > 0 else 0.0
dL_dz3 = dL_dpred3 * dpred_dz3
dL_dw3 = dL_dz3 * x3

print(f"z={z3}  pred={pred3}  loss={loss3}  dL/dw={dL_dw3}")
assert np.isclose(z3, 0.65) and np.isclose(loss3, 0.0225) and np.isclose(dL_dw3, 0.45)
print("Correct!")

<details>
<summary><b>Show solution</b></summary>

```python
x3, w3, b3, target3 = 1.5, 0.3, 0.2, 0.5

z3 = x3 * w3 + b3
pred3 = relu(z3)
loss3 = (pred3 - target3) ** 2

dL_dpred3 = 2 * (pred3 - target3)
dpred_dz3 = 1.0 if z3 > 0 else 0.0
dL_dz3 = dL_dpred3 * dpred_dz3
dL_dw3 = dL_dz3 * x3

print(f"z={z3}  pred={pred3}  loss={loss3}  dL/dw={dL_dw3}")
assert np.isclose(z3, 0.65) and np.isclose(loss3, 0.0225) and np.isclose(dL_dw3, 0.45)
print("Correct!")
```

</details>

## 5. Autograd: the same computation, done by PyTorch

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 REMEMBER THIS — The 'aha' moment</b><br><br>
`loss.backward()` is **not magic**: PyTorch remembers every operation applied
to a tensor with `requires_grad=True` and applies exactly the chain rule we did
by hand in Section 4. Let's rebuild the minimal example (`x=2.0, w=0.5,
target=2.0`) with `torch` and confirm `w.grad` gives the **same -4.0** we
computed by hand.
</div>

In [ ]:
xw = torch.tensor(2.0)
w_t = torch.tensor(0.5, requires_grad=True)  # only w is "trainable"
target_t = torch.tensor(2.0)

pred_t = xw * w_t
loss_t = (pred_t - target_t) ** 2

loss_t.backward()  # applies the chain rule automatically

print("pred:", pred_t.item(), "| loss:", loss_t.item())
print("w.grad (computed by autograd):", w_t.grad.item())
print("dL/dw computed by hand in Section 4:", -4.0)
assert np.isclose(w_t.grad.item(), -4.0)
print("\nThey match exactly! autograd = the chain rule applied by software.")

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 The network is taking shape under your hands.</div>

## 6. Applying it to the real dataset: one loss, one example

We close with the course's running thread: we take **one real row** of the
*Breast Cancer Wisconsin* dataset (30 features), a **fixed** (not trained —
that's Notebook 3) weight vector and a bias we choose, compute the predicted
probability with Sigmoid, and compare our manual Binary Cross-Entropy against
`tf.keras.losses.BinaryCrossentropy()`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)

feature_cols = [c for c in df.columns if c not in ("id", "diagnosis")]
x_row = df.loc[0, feature_cols].to_numpy(dtype="float64")
# diagnosis: 'M' (malignant) -> 1, 'B' (benign) -> 0
y_row = 1.0 if df.loc[0, "diagnosis"] == "M" else 0.0

# Fixed, arbitrary weights (untrained): normalize x so z doesn't explode
x_row_norm = (x_row - x_row.mean()) / x_row.std()
rng = np.random.default_rng(RANDOM_STATE)
w_fixed = rng.normal(0, 0.05, size=x_row_norm.shape)
b_fixed = 0.0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_row = np.dot(x_row_norm, w_fixed) + b_fixed
p_row = sigmoid(z_row)

bce_manual = binary_cross_entropy(np.array([y_row]), np.array([p_row]))[0]
bce_tf = bce_keras(tf.constant([[y_row]]), tf.constant([[p_row]])).numpy()

print("True label (1=malignant, 0=benign):", y_row)
print("Predicted probability (untrained weights):", p_row)
print("Manual BCE:", bce_manual, "| Keras BCE:", bce_tf)
assert np.allclose(bce_manual, bce_tf, atol=1e-5)
print("\nThey match! With untrained weights, this loss is just the starting point.")

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — Why isn't the loss low if the model 'never saw' this data before?</b><br><br>
Because the weights are **random**, not trained — the prediction is essentially
a blind guess. The point of this section isn't to get a low loss, but to
confirm we can **compute it correctly** on a real case. Actually reducing that
loss, by moving the weights along the gradient you learned to compute in
Section 4, is exactly what Notebook 3's optimizer does.
</div>

## 🎯 Self-assessment

Answer without looking back. You don't need perfect phrasing: explain the mechanism in your own words.

**1. Which loss best fits a binary classification problem (malignant or benign?)?**

A. MSE, because it always works equally well
B. Binary Cross-Entropy, because it reflects how correct a probability is
C. No loss is needed once we use Sigmoid
D. Accuracy, because it's the metric that actually matters

<details>
<summary><b>Show answer</b></summary>

**B.** MSE treats errors as numeric distances; BCE is designed for probabilities and strongly penalizes being confidently wrong.

</details>

**2. With y=1, which prediction produces the highest Binary Cross-Entropy?**

A. p = 0.9
B. p = 0.5
C. p = 0.1
D. All produce the same loss

<details>
<summary><b>Show answer</b></summary>

**C.** The further p is from the true label (y=1), the larger -log(p) is. p=0.1 (confidently wrong) gives the highest loss, ≈2.303.

</details>

**3. What exactly does `loss.backward()` compute in PyTorch?**

A. It trains the model and updates the weights directly
B. It applies the chain rule to compute the loss's gradient with respect to every tensor with requires_grad=True
C. It only computes the loss, without gradients
D. It resets the weights to random values

<details>
<summary><b>Show answer</b></summary>

**B.** backward() walks the computational graph backward applying the chain rule at each node, storing the result in .grad — it doesn't move the weights itself (that's the optimizer, Notebook 3).

</details>

**4. If `dL/dw` is exactly 0 for a weight, what does that imply for training at that point?**

A. That the model is now perfect, always
B. That, at that exact point, nudging that weight wouldn't change the loss (it could be a minimum, a maximum, or a saddle point)
C. That there's a code bug and the data must be checked
D. That this weight must be removed from the network

<details>
<summary><b>Show answer</b></summary>

**B.** A zero gradient only says the local slope is flat for that weight at that point — it doesn't guarantee it's the best possible point, or that the rest of the network is optimized too.

</details>

**5. In the graph `w -> (×x) -> pred -> loss`, which rule lets you get `dL/dw` from `dL/dpred` and `dpred/dw`?**

A. L'Hôpital's rule
B. The chain rule: dL/dw = dL/dpred · dpred/dw
C. The rule of three
D. It can't be computed without knowing w

<details>
<summary><b>Show answer</b></summary>

**B.** The chain rule multiplies local derivatives along the path from the loss back to the weight — it's the mathematical basis of backpropagation.

</details>

**6. Does the loss going down on one specific example guarantee the model is better overall?**

<details>
<summary><b>What a good answer should include</b></summary>

- Distinguishes between the loss on a single example and the average (or validation) loss over many examples.
- Mentions that improving on one example can make others worse (overfitting to that point).
- Concludes that only the loss trend over a broad set, not a single example, is a reliable signal.

</details>

**7. Explain the chain rule to a classmate using the x=2, w=0.5 example, without formulas.**

<details>
<summary><b>What a good answer should include</b></summary>

- Describes the path: changing w changes pred, and changing pred changes the loss — the effect 'chains' through.
- Explains the final gradient is the product of how much each step changes times the previous step's change.
- Uses the concrete example (dL/dw = -4.0) to show the result is a computable number, not a vague intuition.

</details>

In [ ]:
celebrate(
    "🎉 Congratulations! You finished Notebook 2: Loss and Backpropagation 🎉",
    "You now know how to measure error with MSE and Cross-Entropy, and how to "
    "compute exactly how you'd move each weight using the chain rule. In Notebook 3 "
    "you'll put that gradient to real use: the optimizer trains the network step by step.",
)